# 02 — Advanced pipeline V2

## 1. Objectif V2
Construire un pipeline avancé pour **Kaggle House Prices** visant une amélioration nette de la baseline V1 : prétraitement métier, features fortes, correction des variables asymétriques, modèles régularisés/boostés et blending simple.

## 2. Rappel score V1
La V1 compare Ridge et RandomForest avec imputation générique et one-hot encoding. Le score CV log RMSE attendu pour une baseline simple se situe autour de `0.145`, selon les données et l'environnement.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = Path("../projects/kaggle-house-prices").resolve()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PROJECT_DIR

## 3. Chargement des données
Les fichiers Kaggle doivent être placés dans `projects/kaggle-house-prices/data/raw/`. Si les données sont absentes, `load_train_test()` lève une erreur claire.

In [ ]:
from src.data import load_train_test

train_df, test_df = load_train_test()
print(train_df.shape, test_df.shape)
train_df.head()

## 4. Analyse rapide des valeurs manquantes métier
On inspecte les colonnes où `NaN` signifie souvent l'absence d'un équipement : garage, sous-sol, piscine, clôture, etc.

In [ ]:
missing_summary = (
    train_df.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame("missing_train")
)
missing_summary["missing_test"] = test_df.isna().sum().reindex(missing_summary.index).fillna(0).astype(int)
missing_summary.head(30)

## 5. Feature engineering
La fonction V2 ajoute des surfaces totales, âges, indicateurs binaires et interactions entre qualité globale et dimensions du bien.

In [ ]:
from src.advanced_features import prepare_advanced_features

X_train, X_test, y = prepare_advanced_features(train_df, test_df)
print(X_train.shape, X_test.shape, y.shape)
X_train.filter(regex="TotalSF|TotalBathrooms|HouseAge|OverallQual", axis=1).head()

## 6. Transformation skew/log1p
Le skew est calculé sur les lignes train uniquement. Les colonnes numériques avec skew `> 0.75` sont transformées par `log1p`, en excluant les variables binaires et ordinales encodées manuellement.

In [ ]:
skewed_cols = X_train.attrs.get("skewed_log1p_columns", [])
print(f"{len(skewed_cols)} colonnes transformées")
skewed_cols

## 7. Comparaison modèles avancés
Modèles obligatoires : Ridge, Lasso, ElasticNet, GradientBoostingRegressor et HistGradientBoostingRegressor. XGBoost/LightGBM sont ajoutés automatiquement s'ils sont installés.

In [ ]:
from src.advanced_model import evaluate_advanced_models

scores = evaluate_advanced_models(X_train, y)
scores

## 8. Sélection du meilleur modèle
Le meilleur modèle single est celui avec le RMSE CV log le plus faible.

In [ ]:
best_name = scores.iloc[0]["model"]
best_score = scores.iloc[0]["cv_rmse_log"]
print(f"Best single model: {best_name} — CV RMSE log: {best_score:.5f}")

## 9. Blending
Un blending pondéré simple est utilisé. Les poids sont normalisés automatiquement et adaptés si XGBoost/LightGBM sont disponibles.

In [ ]:
from src.advanced_model import fit_models_by_names, recommended_blend_weights, blend_predictions

weights = recommended_blend_weights(scores["model"].tolist())
model_names = [name for name in weights if name in set(scores["model"])]
fitted_models = fit_models_by_names(model_names, X_train, y)
blend_log_predictions = blend_predictions(fitted_models, X_test, weights)
print(weights)
blend_log_predictions[:5]

## 10. Génération des fichiers de soumission
Le script complet génère deux CSV : meilleur modèle single et blend.

In [ ]:
from src.submit_advanced import create_advanced_submissions

# Décommenter pour entraîner et écrire les CSV depuis le notebook.
# scores, best_submission, blend_submission = create_advanced_submissions()

## 11. Comparaison V1/V2
Comparer le tableau V2 ci-dessus au score V1 de référence. À compléter après exécution locale avec les données Kaggle.

In [ ]:
comparison = scores.copy()
comparison.loc[len(comparison)] = {"model": "baseline_v1_reference", "cv_rmse_log": 0.145}
comparison.sort_values("cv_rmse_log")

## 12. Limites
- Hyperparamètres initiaux non optimisés par recherche bayésienne.
- Blending simple sans méta-modèle.
- Pas encore de nettoyage spécifique des outliers influents.
- Pas d'analyse SHAP/permutation importance.

## 13. Prochaines itérations
- Ajouter un nettoyage contrôlé des outliers de `GrLivArea`.
- Tuner alpha/l1_ratio et paramètres des modèles boostés.
- Tester un stacking out-of-fold.
- Ajouter des groupes de features par quartier et qualité.
- Renseigner le score Kaggle public après upload.